In [ ]:
# Hello everyone! Please create your own S3 bucket and then change the s3_bucket variable to use this. 
# Then you may work on code to pull it into a sagemaker notebook (follow the class demo) and format it better and pull it into a dataframe. 
# There should be no invalid values but we will check for negatives with code.

import requests
import json
import os
import urllib3
import boto3
import io
import gzip
import logging

In [2]:
s3_client = boto3.client('s3')

bucket_name = "uncc_traffic_data"

In [32]:
ids = ["0600000063", "0600000145", "0600000829", "0600000107", "0600000108", "0600000414", "0600000412", "0600000044", "0600000065", "0600001182"]
id_string_list = ", ".join(f"'{station_id}'" for station_id in ids)
where_clause = f"Location_ID IN ({id_string_list})"
url = "https://services.arcgis.com/NuWFvHYDMVmmxMeM/ArcGIS/rest/services/NCDOT_AADT_Stations/FeatureServer"
query_url = f"{url}/0/query"
s3_bucket = "s3-3601-final-project"
s3_key = "ncdot/traffic/aadt_stations.json.gz"

id_string_list = ", ".join(f"'{sid}'" for sid in ids)
clean_ids = [str(sid).strip().replace("'", "").replace('"', '') for sid in ids]
id_string_list = ", ".join(f"'{sid}'" for sid in clean_ids)
where_clause = f"LocationID IN ({id_string_list})"

session = requests.Session()
all_features = []
spatial_ref = dict()
fields_meta = []

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
})

for i, station_id in enumerate(ids):
    
    params = {
            "where": f"LocationID='{station_id}'",
            "outFields": "LocationID,Route,LOCATION,AADT_2022",
            "f": "json", 
            "returnGeometry": "true"
        }

    try:
        response = session.get(query_url, params=params)
        response.raise_for_status()
        payload = response.json()
        if "error" in payload:
                raise RuntimeError(f"ArcGIS API Error: {payload['error']}")
    except requests.exceptions.JSONDecodeError as e:
        print("\n" + "="*50)
        print("CRITICAL DIAGNOSTIC ERROR DETECTED")
        print("="*50)
        print(f"HTTP Status Received: {response.status_code}")
        print(f"Server Header Content-Type: {response.headers.get('Content-Type')}")
        print("\n--- FIRST 500 CHARACTERS OF RAW SERVER OUTPUT ---")
        print(response.text[:500])
        print("="*50 + "\n")
        raise e
    spatial_ref.update({station_id: payload.get("spatialReference", {})})
    fields_meta.extend(payload.get("fields", []))
    all_features.extend(payload.get("features", []))
    
    



output_data = {
    "spatialReference": spatial_ref,
    "fields": fields_meta,
    "features": all_features
}


compressed = io.BytesIO()
with gzip.GzipFile(fileobj=compressed, mode="wb") as gz:
    gz.write(json.dumps(output_data).encode("utf-8"))
                                    
s3_client.put_object(
    Bucket=s3_bucket,
    Key=s3_key,
    Body=compressed.getvalue(),
    ContentType="application/json",
    ContentEncoding="gzip",
)

{'ResponseMetadata': {'RequestId': '9YNX6N8NG3Z5EBWG',
  'HostId': 'Nkd8OEZYWJLX5wuKdu6K+62FmHIuv9jDEtq0NNnbh4E5+V3dkuaTLRpCxMikbEuY2ryUve1KqWs=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'Nkd8OEZYWJLX5wuKdu6K+62FmHIuv9jDEtq0NNnbh4E5+V3dkuaTLRpCxMikbEuY2ryUve1KqWs=',
   'x-amz-request-id': '9YNX6N8NG3Z5EBWG',
   'date': 'Tue, 22 Sep 2026 23:14:04 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"6a842976ac264e64482063363c12f163"',
   'x-amz-checksum-crc32': 'ahIdUg==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"6a842976ac264e64482063363c12f163"',
 'ChecksumCRC32': 'ahIdUg==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}